# MEGAN 数据处理教程

## 将化学反应转化为图编辑序列

本 Notebook 是 MEGAN 教程的第二部分，展示 MEGAN 如何将原始的化学反应 SMILES 数据转化为模型可训练的**图编辑序列 (Graph Edit Sequence)** 样本。

### 数据处理总流程

```
原始反应 CSV (product >> reactants)
    ↓ Step 1: SMILES 解析与原子映射对齐
Mol 对象 (product_mol, reactants_mol)
    ↓ Step 2: 分子图构建 (含超节点 supernode)
邻接矩阵 adj + 节点特征 nodes
    ↓ Step 3: 图编辑动作提取 (product → reactants 的差异)
动作序列 [BondEdit, AtomEdit, AddAtom, ..., Stop]
    ↓ Step 4: 训练样本生成 (每步一个样本)
(graph_state_i, action_i) 对
    ↓ Step 5: 特征稀疏化与存储
nodes.npz + adj.npz + sample_data.npz + metadata.csv
```

### 核心思想

与基于模板的方法不同，MEGAN **不需要提取反应模板**，而是通过比较产物和反应物的分子图差异，自动生成一系列图编辑操作。模型学习的是**如何一步步编辑产物分子图使其变成反应物**。

> **使用数据**: 本教程使用微型演示数据集 `data/demo_data.csv`（来自 USPTO 专利的真实反应数据）

In [1]:
# ============================================================
# 导入依赖库并配置路径
# ============================================================
import sys
import csv
import copy
import random
from pathlib import Path
from typing import Tuple, List, Optional

import numpy as np
import pandas as pd

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Draw, rdchem
from rdkit.Chem.rdchem import RWMol, Mol, BondType, ChiralType, BondStereo
from rdkit.Chem.Draw import IPythonConsole

# 禁用 RDKit 警告（氢原子相关的警告无影响）
RDLogger.DisableLog('rdApp.*')

# 配置路径
TUTORIAL_DIR = Path(".").resolve()
PROJECT_ROOT = TUTORIAL_DIR.parents[3]
MEGAN_SRC_DIR = PROJECT_ROOT / "source_repos" / "megan"
DATA_DIR = TUTORIAL_DIR / "data"

print(f"项目根目录: {PROJECT_ROOT}")
print(f"MEGAN 源码: {MEGAN_SRC_DIR}")
print(f"数据目录:   {DATA_DIR}")

项目根目录: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis
MEGAN 源码: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/source_repos/megan
数据目录:   /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/teaching_demos/2.single_step_retro_tutorial/2.2.template_free/2.2.1.megan/data


## Step 1: 原始数据加载与探索

我们使用与 GLN 教程相同的 `demo_data.csv`，其格式为：
```
PatentNumber, class, reactants>reagents>production
```
每条数据是一个带**原子映射编号**的化学反应 SMILES。

In [2]:
# ============================================================
# 加载演示数据集
# ============================================================
reactions = []
with open(DATA_DIR / "demo_data.csv", "r") as f:
    reader = csv.reader(f)
    header = next(reader)
    for row in reader:
        if len(row) >= 3:
            patent, rxn_class, rxn_smiles = row[0], row[1], row[2]
            # 拆分反应 SMILES: reactants>reagents>product
            parts = rxn_smiles.split(">")
            if len(parts) == 3:
                reactants_smi, reagents_smi, product_smi = parts
                reactions.append({
                    'patent': patent.strip(),
                    'reactants': reactants_smi.strip(),
                    'reagents': reagents_smi.strip(),
                    'product': product_smi.strip(),
                    'full_rxn': rxn_smiles.strip()
                })

print(f"加载了 {len(reactions)} 条反应数据")
print(f"\n{'='*60}")
print("前 3 条反应数据示例:")
print(f"{'='*60}")
for i, rxn in enumerate(reactions[:3]):
    print(f"\n反应 {i+1} (专利号: {rxn['patent']}):")
    print(f"  产物:   {rxn['product'][:80]}...")
    print(f"  反应物: {rxn['reactants'][:80]}...")

加载了 1000 条反应数据

前 3 条反应数据示例:

反应 1 (专利号: US08263610B2):
  产物:   [CH3:1][c:2]1[n:3][c:4]2[c:5]([s:6]1)[C:7](=[O:8])/[C:9](=[CH:10]\[N:11]([CH3:12...
  反应物: C1CCOC1.CCOCC.Cc1ccccc1.O=[CH:10][CH:9]1[C:7](=[O:8])[c:5]2[c:4]([n:3][c:2]([CH3...

反应 2 (专利号: US05378712):
  产物:   [O:1]=[C:2]([NH:3][CH:4]1[CH:5]=[CH:6][CH:7]([OH:8])[CH2:9][CH2:10]1)[O:11][CH2:...
  反应物: CCN(CC)CC.CCOCC.CN(C)C=O.O=C1CCC(=O)N1O[C:2](=[O:1])[O:11][CH2:12][c:13]1[cH:14]...

反应 3 (专利号: US04136194):
  产物:   [CH3:1][c:2]1[cH:3][cH:4][c:5]([N:6]=[CH:7]/[CH:8]=[CH:9]/[N:10]([C:11](=[O:12])...
  反应物: Cc1c(CC(=O)Cl)[c:26]2[c:5](n1[C:11](=[O:12])[c:13]1[cH:14][cH:15][c:16](Cl)[cH:1...


## Step 2: SMILES 解析与原子映射对齐

MEGAN 的数据处理第一步是将 SMILES 字符串解析为 RDKit 分子对象，并进行**原子映射对齐**。

### 为什么需要原子映射？

在逆合成分析中，我们需要知道产物中的每个原子对应反应物中的哪个原子。原子映射编号（如 `:1`, `:2`）提供了这种对应关系。

### 关键处理步骤
1. **`fix_incomplete_mappings()`**: 为没有映射编号的原子补充编号
2. **`reac_to_canonical()`**: 将分子转换为标准化形式，确保原子序一致
3. **`fix_explicit_hs()`**: 处理氢原子的隐式/显式转换问题
4. **`renumber_atoms_for_mapping()`**: 让原子在分子列表中按映射编号排序

In [3]:
# ============================================================
# SMILES 预处理工具函数
# 来源: MEGAN src/feat/utils.py 和 src/utils/__init__.py
# ============================================================

def fix_incomplete_mappings(sub_mol: Mol, prod_mol: Mol) -> Tuple[Mol, Mol]:
    """
    为没有原子映射编号的原子补充编号。
    确保产物和反应物中每个原子都有唯一的映射号。
    """
    max_map = max(a.GetAtomMapNum() for a in sub_mol.GetAtoms())
    max_map = max(max(a.GetAtomMapNum() for a in prod_mol.GetAtoms()), max_map)
    
    for mol in (sub_mol, prod_mol):
        for a in mol.GetAtoms():
            map_num = a.GetAtomMapNum()
            if map_num is None or map_num < 1:
                max_map += 1
                a.SetAtomMapNum(max_map)
    return sub_mol, prod_mol


def reac_to_canonical(sub_mol, prod_mol):
    """
    将分子转换为标准化形式：
    - 先转为 SMILES 再解析回来，使原子序为 RDKit 标准序
    - 重新编号映射号使产物的映射号与原子列表序一致
    """
    sub_mol = Chem.MolFromSmiles(Chem.MolToSmiles(sub_mol))
    prod_mol = Chem.MolFromSmiles(Chem.MolToSmiles(prod_mol))
    
    map2map = {}
    for i, a in enumerate(prod_mol.GetAtoms()):
        map2map[a.GetAtomMapNum()] = i + 1
        a.SetAtomMapNum(i + 1)
    
    max_map = max(map2map.values())
    for a in sub_mol.GetAtoms():
        m = a.GetAtomMapNum()
        if m in map2map:
            a.SetAtomMapNum(map2map[m])
        else:
            max_map += 1
            a.SetAtomMapNum(max_map)
    
    return sub_mol, prod_mol


def fix_explicit_hs(mol: Mol) -> Mol:
    """修复 RDKit 中隐式/显式氢原子的问题"""
    for a in mol.GetAtoms():
        a.SetNoImplicit(False)
    mol = Chem.AddHs(mol, explicitOnly=True)
    mol = Chem.RemoveHs(mol)
    Chem.SanitizeMol(mol)
    return mol


def renumber_atoms_for_mapping(mol: Mol) -> Mol:
    """重新排列原子，使原子列表中的顺序与映射编号一致"""
    new_order = [a.GetAtomMapNum() for a in mol.GetAtoms()]
    new_order = [int(a) for a in np.argsort(new_order)]
    return Chem.RenumberAtoms(mol, new_order)


def complete_mappings(subs: str, prod: str) -> Tuple[str, str]:
    """为不完整映射的反应添加映射编号"""
    subs_mol = Chem.MolFromSmiles(subs)
    prod_mol = Chem.MolFromSmiles(prod)
    
    max_num = 0
    for mol in (subs_mol, prod_mol):
        max_num = max(max(a.GetAtomMapNum() for a in mol.GetAtoms()), max_num)
    
    curr_num = max_num + 1
    for mol in (subs_mol, prod_mol):
        for a in mol.GetAtoms():
            if a.GetAtomMapNum() == 0:
                a.SetAtomMapNum(curr_num)
                curr_num += 1
    
    return Chem.MolToSmiles(subs_mol), Chem.MolToSmiles(prod_mol)


# ============================================================
# 演示: 处理一条反应
# ============================================================
rxn = reactions[0]
print(f"原始反应 (专利: {rxn['patent']}):")
print(f"  产物:   {rxn['product'][:80]}")
print(f"  反应物: {rxn['reactants'][:80]}")

# 补全映射
reactants_smi, product_smi = complete_mappings(rxn['reactants'], rxn['product'])

# 解析为 Mol 对象
prod_mol = Chem.MolFromSmiles(product_smi)
sub_mol = Chem.MolFromSmiles(reactants_smi)

# 对齐处理
sub_mol, prod_mol = fix_incomplete_mappings(sub_mol, prod_mol)
sub_mol, prod_mol = reac_to_canonical(sub_mol, prod_mol)
sub_mol = fix_explicit_hs(sub_mol)
prod_mol = fix_explicit_hs(prod_mol)
sub_mol = renumber_atoms_for_mapping(sub_mol)
prod_mol = renumber_atoms_for_mapping(prod_mol)

print(f"\n标准化后:")
print(f"  产物原子数: {prod_mol.GetNumAtoms()}, 键数: {prod_mol.GetNumBonds()}")
print(f"  反应物原子数: {sub_mol.GetNumAtoms()}, 键数: {sub_mol.GetNumBonds()}")

# 展示原子映射对应关系
prod_maps = set(a.GetAtomMapNum() for a in prod_mol.GetAtoms())
sub_maps = set(a.GetAtomMapNum() for a in sub_mol.GetAtoms())
print(f"\n原子映射分析:")
print(f"  产物映射号: {sorted(prod_maps)[:10]}... (共 {len(prod_maps)} 个)")
print(f"  反应物映射号: {sorted(sub_maps)[:10]}... (共 {len(sub_maps)} 个)")
print(f"  共有原子(产物∩反应物): {len(prod_maps & sub_maps)} 个")
print(f"  仅在反应物中(需要添加): {len(sub_maps - prod_maps)} 个")

原始反应 (专利: US08263610B2):
  产物:   [CH3:1][c:2]1[n:3][c:4]2[c:5]([s:6]1)[C:7](=[O:8])/[C:9](=[CH:10]\[N:11]([CH3:12
  反应物: C1CCOC1.CCOCC.Cc1ccccc1.O=[CH:10][CH:9]1[C:7](=[O:8])[c:5]2[c:4]([n:3][c:2]([CH3

标准化后:
  产物原子数: 15, 键数: 16
  反应物原子数: 33, 键数: 32

原子映射分析:
  产物映射号: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]... (共 15 个)
  反应物映射号: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]... (共 33 个)
  共有原子(产物∩反应物): 15 个
  仅在反应物中(需要添加): 18 个


## Step 3: 构建原子特征与键特征

MEGAN 使用 **one-hot 离散编码** 来表示原子和键的属性。每个原子/键用一组离散特征编码，然后通过 `ravel_multi_index` 压缩为单个整数存储。

### 原子特征 (8 个属性)

| 属性 | 含义 | 可能取值 |
|------|------|---------|
| `is_supernode` | 是否为超节点 | 0, 1 |
| `atomic_num` | 原子序号 | H(1), C(6), N(7), O(8), ... |
| `formal_charge` | 形式电荷 | -1, 0, +1, +2, +3 |
| `chiral_tag` | 手性标记 | 无/CW/CCW |
| `num_explicit_hs` | 显式氢原子数 | 0, 1, 2, 4 |
| `is_aromatic` | 是否芳香 | 0, 1 |
| `is_edited` | 是否被编辑过 | 0, 1 |
| `is_reactant` | 是否为反应物原子 | 0, 1 |

### 键特征 (3 个属性)

| 属性 | 含义 | 可能取值 |
|------|------|---------|
| `bond_type` | 键类型 | self, supernode, 单键(1), 双键(2), 三键(3), 芳香键(12) |
| `bond_stereo` | 键立体化学 | 无(0), E(2), Z(3) |
| `is_edited` | 是否被编辑过 | 0, 1 |

In [4]:
# ============================================================
# 原子/键特征定义与编码
# 来源: MEGAN src/feat/graph_features.py, src/feat/__init__.py
# ============================================================

# 原子特征键名（有序）
ORDERED_ATOM_OH_KEYS = [
    'is_supernode', 'atomic_num', 'formal_charge', 'chiral_tag',
    'num_explicit_hs', 'is_aromatic', 'is_edited', 'is_reactant'
]

# 键特征键名（有序）
ORDERED_BOND_OH_KEYS = ['bond_type', 'bond_stereo', 'is_edited']

# 可编辑的原子属性（MEGAN 的 AtomEditAction 可以修改这些属性）
ATOM_EDIT_TUPLE_KEYS = ['formal_charge', 'chiral_tag', 'num_explicit_hs', 'is_aromatic']

# 原子属性的可能取值（来自 USPTO 训练集的统计）
ATOM_PROPS = {
    'atomic_num': [1, 3, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 19, 22,
                   29, 30, 32, 33, 34, 35, 47, 50, 51, 52, 53, 82, 83],
    'formal_charge': [-1, 0, 1, 2, 3],
    'chiral_tag': [0, 1, 2],
    'num_explicit_hs': [0, 1, 2, 4],
    'is_aromatic': [0, 1],
    'is_supernode': [0, 1],
    'is_edited': [0, 1],
    'is_reactant': [0, 1]
}

# 键属性的可能取值
BOND_PROPS = {
    'bond_type': ['self', 'supernode', 1, 2, 3, 12],  # 1=单键, 2=双键, 3=三键, 12=芳香键
    'bond_stereo': [0, 2, 3],
    'is_edited': [0, 1],
}

# 属性值 → one-hot 编号的映射字典
# 0 保留作为"未知/缺失"值，实际编号从 1 开始
ATOM_PROP2OH = {k: {ap: i + 1 for i, ap in enumerate(vals)} for k, vals in ATOM_PROPS.items()}
BOND_PROP2OH = {k: {ap: i + 1 for i, ap in enumerate(vals)} for k, vals in BOND_PROPS.items()}

# 添加特殊键类型的映射
BOND_PROP2OH['bond_type']['self'] = BOND_PROPS['bond_type'].index('self') + 1
BOND_PROP2OH['bond_type']['supernode'] = BOND_PROPS['bond_type'].index('supernode') + 1

print("=== 原子特征编码映射 ===")
for key, mapping in ATOM_PROP2OH.items():
    print(f"  {key:20s}: {len(mapping)} 种取值 → 编号 1~{len(mapping)}")

print(f"\n=== 键特征编码映射 ===")
for key, mapping in BOND_PROP2OH.items():
    print(f"  {key:20s}: {len(mapping)} 种取值 → 编号 1~{len(mapping)}")


# ============================================================
# 特征提取函数
# ============================================================
def get_atom_features(atom, atom_oh_keys=ORDERED_ATOM_OH_KEYS, atom_prop2oh=ATOM_PROP2OH):
    """从 RDKit Atom 对象提取特征向量"""
    result = []
    for key in atom_oh_keys:
        if key not in atom_prop2oh:
            continue
        # 获取原子属性值
        if key == 'is_supernode':
            val = 0
        elif key == 'is_edited':
            val = 1 if (atom.HasProp('is_edited') and atom.GetBoolProp('is_edited')) else 0
        elif key == 'is_reactant':
            val = 1 if (atom.HasProp('in_target') and atom.GetBoolProp('in_target')) else 0
        elif key == 'atomic_num':
            val = atom.GetAtomicNum()
        elif key == 'formal_charge':
            val = atom.GetFormalCharge()
        elif key == 'chiral_tag':
            val = int(atom.GetChiralTag())
        elif key == 'num_explicit_hs':
            val = atom.GetNumExplicitHs()
        elif key == 'is_aromatic':
            val = int(atom.GetIsAromatic())
        else:
            val = 0
        
        oh_val = atom_prop2oh[key].get(val, 0)
        result.append(oh_val)
    return result


def get_bond_features(bond, bond_oh_keys=ORDERED_BOND_OH_KEYS, bond_prop2oh=BOND_PROP2OH):
    """从 RDKit Bond 对象提取特征向量"""
    result = []
    for key in bond_oh_keys:
        if key not in bond_prop2oh:
            continue
        if key == 'bond_type':
            val = int(bond.GetBondType())
        elif key == 'bond_stereo':
            val = int(bond.GetStereo())
        elif key == 'is_edited':
            val = 1 if (bond.HasProp('is_edited') and bond.GetBoolProp('is_edited')) else 0
        else:
            val = 0
        
        oh_val = bond_prop2oh[key].get(val, 0)
        result.append(oh_val)
    return result


# 演示: 提取一个碳原子的特征
demo_mol = Chem.MolFromSmiles("[CH3:1][OH:2]")
atom = demo_mol.GetAtomWithIdx(0)  # 碳原子
features = get_atom_features(atom)
print(f"\n=== 特征提取演示 ===")
print(f"原子: {atom.GetSymbol()} (映射号 {atom.GetAtomMapNum()})")
print(f"特征向量: {features}")
print(f"对应特征键: {[k for k in ORDERED_ATOM_OH_KEYS if k in ATOM_PROP2OH]}")

=== 原子特征编码映射 ===
  atomic_num          : 29 种取值 → 编号 1~29
  formal_charge       : 5 种取值 → 编号 1~5
  chiral_tag          : 3 种取值 → 编号 1~3
  num_explicit_hs     : 4 种取值 → 编号 1~4
  is_aromatic         : 2 种取值 → 编号 1~2
  is_supernode        : 2 种取值 → 编号 1~2
  is_edited           : 2 种取值 → 编号 1~2
  is_reactant         : 2 种取值 → 编号 1~2

=== 键特征编码映射 ===
  bond_type           : 6 种取值 → 编号 1~6
  bond_stereo         : 3 种取值 → 编号 1~3
  is_edited           : 2 种取值 → 编号 1~2

=== 特征提取演示 ===
原子: C (映射号 1)
特征向量: [1, 4, 2, 1, 0, 1, 1, 1]
对应特征键: ['is_supernode', 'atomic_num', 'formal_charge', 'chiral_tag', 'num_explicit_hs', 'is_aromatic', 'is_edited', 'is_reactant']


## Step 4: 构建分子图（含超节点 Supernode）

MEGAN 的分子图表示有一个重要特点：在普通原子节点之外，额外引入一个**超节点 (Supernode)**，编号为 0。

### 超节点的作用
- 超节点连接所有原子，提供**全局信息交换**通道
- 在推理时，超节点用于发出 **Stop 动作**（终止编辑序列）
- 超节点与原子之间的边使用特殊的 `supernode` 键类型

### 分子图结构
```
           Supernode (编号 0)
          / | | | \
         /  | | |  \
        原子1 原子2 ... 原子n
         \  /      /
          化学键连接
```

- **自环 (self-loop)**: 每个节点都有与自身的自环边（键类型为 `self`）
- **对称邻接矩阵**: `adj[i,j] = adj[j,i]` 对于所有化学键

In [5]:
# ============================================================
# Step 4: 构建分子图（含超节点 Supernode）
# ============================================================

def ravel_atom_features(atom_features, node_oh_dim):
    """将多维原子特征索引压缩为单一整数"""
    return int(np.ravel_multi_index(atom_features, node_oh_dim))

def unravel_atom_features(atom_features, node_oh_dim):
    """从单一整数恢复多维原子特征索引"""
    return np.array(np.unravel_index(atom_features, node_oh_dim))

def ravel_bond_features(bond_features, edge_oh_dim):
    """将多维键特征索引压缩为单一整数"""
    return int(np.ravel_multi_index(bond_features, edge_oh_dim))

def unravel_bond_features(bond_features, edge_oh_dim):
    """从单一整数恢复多维键特征索引"""
    return np.array(np.unravel_index(bond_features, edge_oh_dim))


def get_graph(mol, ravel=True, to_array=False,
              atom_feature_keys=ORDERED_ATOM_OH_KEYS,
              bond_feature_keys=ORDERED_BOND_OH_KEYS,
              atom_prop2oh=ATOM_PROP2OH,
              bond_prop2oh=BOND_PROP2OH):
    """
    将 RDKit 分子对象转换为 MEGAN 的图表示。
    
    参数:
        mol: RDKit Mol 对象 (需要 atom map numbers)
        ravel: 是否将多维特征压缩为单一整数
        to_array: 是否返回密集邻接矩阵（否则返回 COO 格式）
    
    返回:
        (adj, nodes): 邻接矩阵和节点特征
    """
    # 计算特征维度
    self_bond_features = np.zeros(len(bond_feature_keys), dtype=int)
    edge_oh_dim = [len(bond_prop2oh[k]) + 1 for k in bond_feature_keys if k in bond_prop2oh]
    node_oh_dim = [len(atom_prop2oh[k]) + 1 for k in atom_feature_keys if k in atom_prop2oh]
    
    # 超节点和自环的特殊特征
    supernode_bond_features = np.zeros(len(edge_oh_dim), dtype=int)
    supernode_atom_features = np.zeros(len(node_oh_dim), dtype=int)
    
    for i, k in enumerate(bond_feature_keys):
        if k == 'bond_type':
            self_bond_features[i] = bond_prop2oh['bond_type']['self']
            supernode_bond_features[i] = bond_prop2oh['bond_type']['supernode']
    
    for i, k in enumerate(atom_feature_keys):
        if k == 'is_supernode':
            supernode_atom_features[i] = atom_prop2oh['is_supernode'][1]
    
    # 初始化图结构
    adj_vals, adj_rows, adj_cols = [], [], []
    n_nodes = max(a.GetAtomMapNum() for a in mol.GetAtoms()) + 1
    nodes = np.zeros((n_nodes, len(supernode_atom_features)), dtype=int)
    nodes[0] = supernode_atom_features  # 超节点始终为编号 0

    NOT_IN_REACTANT = ATOM_PROP2OH['is_reactant'][0]
    IN_REACTANT = ATOM_PROP2OH['is_reactant'][1]
    is_reactant = np.full((n_nodes,), dtype=int, fill_value=NOT_IN_REACTANT)
    
    def add_bond(i1, i2, val):
        """添加边（对称）"""
        adj_vals.append(val)
        adj_rows.append(i1)
        adj_cols.append(i2)
        if i1 != i2:
            adj_vals.append(val)
            adj_rows.append(i2)
            adj_cols.append(i1)
    
    # 超节点自环
    compounds = {0: {0}}
    is_reactant[0] = 0
    add_bond(0, 0, self_bond_features)
    
    # 遍历所有原子
    for a in mol.GetAtoms():
        i = a.GetAtomMapNum()
        compounds[i] = {i}
        if a.HasProp('in_target') and a.GetBoolProp('in_target'):
            is_reactant[i] = IN_REACTANT
        
        # 原子特征
        nodes[i] = get_atom_features(a, atom_feature_keys, atom_prop2oh=atom_prop2oh)
        
        # 超节点-原子边 + 自环
        add_bond(0, i, supernode_bond_features)
        add_bond(i, i, self_bond_features)
    
    # 遍历所有化学键
    for bond in mol.GetBonds():
        a1 = bond.GetBeginAtom().GetAtomMapNum()
        a2 = bond.GetEndAtom().GetAtomMapNum()
        bond_features = get_bond_features(bond, bond_feature_keys, bond_prop2oh=bond_prop2oh)
        add_bond(a1, a2, bond_features)
        
        # 合并 compound 集合
        if compounds[a1] is not compounds[a2]:
            common = compounds[a1].union(compounds[a2])
            for m in common:
                compounds[m] = common
    
    # 编号独立化合物
    cur_mol = 1
    mol_inds = np.zeros((n_nodes,), dtype=int)
    for i in range(1, n_nodes):
        if i in compounds and mol_inds[i] == 0:
            for m in compounds[i]:
                mol_inds[m] = cur_mol
            if any(is_reactant[m] == IN_REACTANT for m in compounds[i]):
                for m in compounds[i]:
                    is_reactant[m] = IN_REACTANT
            cur_mol += 1
    
    # 写入 is_reactant 特征
    if 'is_reactant' in atom_feature_keys and 'is_reactant' in atom_prop2oh:
        is_reactant_feat_ind = atom_feature_keys.index('is_reactant')
        nodes[:, is_reactant_feat_ind] = is_reactant
    
    # 压缩特征
    if ravel:
        nodes = np.array([ravel_atom_features(node, node_oh_dim) for node in nodes])
        adj_vals = [ravel_bond_features(val, edge_oh_dim) for val in adj_vals]
    
    if to_array:
        if ravel:
            adj = np.zeros((n_nodes, n_nodes), dtype=int)
        else:
            adj = np.zeros((n_nodes, n_nodes, len(edge_oh_dim)), dtype=int)
        for val, row, col in zip(adj_vals, adj_rows, adj_cols):
            adj[row, col] = val
        return adj, nodes
    
    return (adj_vals, adj_rows, adj_cols), nodes


# === 演示 ===
print("=" * 60)
print("Step 4: 分子图构建演示")
print("=" * 60)

# 用一个简单的分子演示
demo_smi = "[CH3:1][OH:2]"  # 甲醇
mol = Chem.MolFromSmiles(demo_smi)

# 1) 非压缩模式：查看完整特征
adj_array, nodes_array = get_graph(mol, ravel=False, to_array=True)
print(f"\n分子: {demo_smi}")
print(f"节点数（含超节点）: {len(nodes_array)}")
print(f"节点特征形状: {nodes_array.shape}  (n_nodes × n_atom_features)")
print(f"邻接矩阵形状: {adj_array.shape}  (n_nodes × n_nodes × n_bond_features)")

print(f"\n节点 0（超节点）特征: {nodes_array[0]}")
print(f"节点 1（C, map=1）特征: {nodes_array[1]}")
print(f"节点 2（O, map=2）特征: {nodes_array[2]}")

print(f"\n邻接矩阵切片 (bond_type 维度):")
print(adj_array[:, :, 0])

# 2) 压缩模式：用于实际训练
adj_coo, nodes_raveled = get_graph(mol, ravel=True, to_array=False)
print(f"\n压缩后节点特征（整数编码）: {nodes_raveled}")
print(f"COO 邻接: {len(adj_coo[0])} 条边")

# 3) 验证超节点连接性
print(f"\n超节点连接验证（邻接矩阵第0行非零列）:")
nonzero_cols = np.where(adj_array[0, :, 0] > 0)[0]
print(f"  超节点连接的节点: {nonzero_cols}")  # 应该包含所有原子节点

print("\n✅ 分子图构建完成！超节点连接所有原子，每个节点有自环。")

Step 4: 分子图构建演示

分子: [CH3:1][OH:2]
节点数（含超节点）: 3
节点特征形状: (3, 8)  (n_nodes × n_atom_features)
邻接矩阵形状: (3, 3, 3)  (n_nodes × n_nodes × n_bond_features)

节点 0（超节点）特征: [2 0 0 0 0 0 0 0]
节点 1（C, map=1）特征: [1 4 2 1 0 1 1 1]
节点 2（O, map=2）特征: [1 6 2 1 2 1 1 1]

邻接矩阵切片 (bond_type 维度):
[[1 2 2]
 [2 1 3]
 [2 3 1]]

压缩后节点特征（整数编码）: [194400 111388 117922]
COO 邻接: 9 条边

超节点连接验证（邻接矩阵第0行非零列）:
  超节点连接的节点: [0 1 2]

✅ 分子图构建完成！超节点连接所有原子，每个节点有自环。


## Step 5: 提取图编辑动作序列（MEGAN 核心创新）

这是 MEGAN 数据处理的**核心步骤**。目标是：通过对比**产物**与**反应物**的分子图差异，自动提取出一系列**图编辑动作 (Graph Edit Actions)**。

### 动作类型

| 动作类型 | 说明 | 示例 |
|---------|------|------|
| `AtomEditAction` | 修改原子属性（电荷、手性、氢数、芳香性） | 原子的 formal_charge 从 0 变为 -1 |
| `BondEditAction` | 修改/删除/添加化学键 | 单键变双键、删除键 |
| `AddAtomAction` | 添加新原子（及连接键） | 添加一个甲基基团 |
| `AddRingAction` | 添加苯环 | 添加整个苯环结构 |
| `StopAction` | 终止编辑序列 | 编辑完成 |

### 动作优先级（逆合成方向）
```
deleted_bonds → added_bonds → changed_bonds → changed_atoms → new_rings → new_atoms
```
先处理键的删除（断开反应键），再处理键的添加和修改，最后处理原子变化。

### 工作流程
```
产物分子 ----对比----> 反应物分子
     |                    |
     |  找出所有差异       |
     |  按优先级排序       |
     v                    v
   动作1 → 动作2 → ... → Stop
```

每执行一个动作后，分子和图都会被**立即更新**（apply + graph_apply），确保下一步动作基于最新状态。

In [6]:
# ============================================================
# Step 5: 定义图编辑动作类
# ============================================================
# 这些类描述了产物→反应物的每一步编辑操作

from rdkit.Chem import BondType, rdchem
from abc import ABCMeta, abstractmethod

# 原子编辑属性键
ATOM_EDIT_TUPLE_KEYS = ['formal_charge', 'chiral_tag', 'num_explicit_hs', 'is_aromatic']

# 构建动作词汇表（action_vocab）
# 这是 MEGAN 训练所需的核心数据结构
action_vocab = {
    'prop2oh': {
        'atom': ATOM_PROP2OH,
        'bond': BOND_PROP2OH,
    },
    'atom_feat_ind': {k: i for i, k in enumerate(ORDERED_ATOM_OH_KEYS)},
    'bond_feat_ind': {k: i for i, k in enumerate(ORDERED_BOND_OH_KEYS)},
}
print("动作词汇表键:")
for k, v in action_vocab.items():
    if isinstance(v, dict):
        print(f"  {k}: {list(v.keys())}")


def get_atom_ind(mol, atom_map):
    """根据 atom map number 查找原子索引"""
    for a in mol.GetAtoms():
        if a.GetAtomMapNum() == atom_map:
            return a.GetIdx()
    return None


def atom_to_edit_tuple(atom):
    """提取原子的可编辑属性元组"""
    return (atom.GetFormalCharge(), int(atom.GetChiralTag()),
            atom.GetNumExplicitHs(), int(atom.GetIsAromatic()))


def get_bond_tuple(bond):
    """提取键的原子对和属性元组"""
    a1 = bond.GetBeginAtom().GetAtomMapNum()
    a2 = bond.GetEndAtom().GetAtomMapNum()
    if a1 > a2:
        a1, a2 = a2, a1
    return (a1, a2, int(bond.GetBondType()), int(bond.GetStereo()))


class ReactionAction(metaclass=ABCMeta):
    """图编辑动作基类"""
    def __init__(self, atom_map1, atom_map2, is_hard=False):
        self.atom_map1 = atom_map1
        self.atom_map2 = atom_map2
        self.is_hard = is_hard
    
    @abstractmethod
    def get_tuple(self):
        raise NotImplementedError
    
    @abstractmethod
    def apply(self, mol):
        """在 RDKit 分子上执行动作"""
        raise NotImplementedError
    
    @abstractmethod
    def graph_apply(self, adj, nodes):
        """在图矩阵上执行动作"""
        raise NotImplementedError


class StopAction(ReactionAction):
    """终止编辑序列"""
    def __init__(self):
        super().__init__(0, -1)
    
    def get_tuple(self):
        return ('stop',)
    
    def apply(self, mol):
        return mol
    
    def graph_apply(self, adj, nodes):
        return adj, nodes
    
    def __str__(self):
        return "Stop"


class AtomEditAction(ReactionAction):
    """修改原子属性"""
    def __init__(self, atom_map1, formal_charge, chiral_tag, num_explicit_hs, is_aromatic, is_hard=False):
        super().__init__(atom_map1, -1, is_hard)
        self.formal_charge = formal_charge
        self.chiral_tag = chiral_tag
        self.num_explicit_hs = num_explicit_hs
        self.is_aromatic = is_aromatic
    
    def get_tuple(self):
        return ('change_atom', (self.formal_charge, self.chiral_tag, self.num_explicit_hs, self.is_aromatic))
    
    def apply(self, mol):
        atom_ind = get_atom_ind(mol, self.atom_map1)
        atom = mol.GetAtomWithIdx(atom_ind)
        atom.SetFormalCharge(self.formal_charge)
        atom.SetChiralTag(rdchem.ChiralType.values[self.chiral_tag])
        atom.SetNumExplicitHs(self.num_explicit_hs)
        atom.SetIsAromatic(self.is_aromatic)
        atom.SetBoolProp('is_edited', True)
        return mol
    
    def graph_apply(self, adj, nodes):
        nodes = nodes.copy()
        feat_keys = ATOM_EDIT_TUPLE_KEYS + ['is_edited']
        feat_vals = (self.formal_charge, self.chiral_tag, self.num_explicit_hs, self.is_aromatic, 1)
        for key, val in zip(feat_keys, feat_vals):
            ind = action_vocab['atom_feat_ind'].get(key, -1)
            if ind != -1:
                nodes[self.atom_map1, ind] = ATOM_PROP2OH[key].get(val, 0)
        return adj, nodes
    
    def __str__(self):
        return f"EditAtom({self.atom_map1}): charge={self.formal_charge}, hs={self.num_explicit_hs}"


class BondEditAction(ReactionAction):
    """修改/添加/删除化学键"""
    def __init__(self, atom_map1, atom_map2, bond_type, bond_stereo, is_hard=False):
        super().__init__(atom_map1, atom_map2, is_hard)
        self.bond_type = bond_type    # None 表示删除键
        self.bond_stereo = bond_stereo
    
    def get_tuple(self):
        if self.bond_type is None:
            return ('delete_bond',)
        return ('change_bond', (self.bond_type, self.bond_stereo))
    
    def apply(self, mol):
        ai1 = get_atom_ind(mol, self.atom_map1)
        ai2 = get_atom_ind(mol, self.atom_map2)
        if self.bond_type is None:
            # 删除键
            mol.RemoveBond(ai1, ai2)
        else:
            bond = mol.GetBondBetweenAtoms(ai1, ai2)
            if bond is None:
                # 添加新键
                b_type = rdchem.BondType.values[self.bond_type]
                bond_idx = mol.AddBond(ai1, ai2, order=b_type) - 1
                bond = mol.GetBondWithIdx(bond_idx)
            else:
                # 修改现有键
                bond.SetBondType(rdchem.BondType.values[self.bond_type])
            bond.SetStereo(rdchem.BondStereo.values[self.bond_stereo])
            bond.SetBoolProp('is_edited', True)
        return mol
    
    def graph_apply(self, adj, nodes):
        adj = adj.copy()
        if self.bond_type is None:
            adj[self.atom_map1, self.atom_map2] = np.zeros(adj.shape[2], dtype=int)
            adj[self.atom_map2, self.atom_map1] = np.zeros(adj.shape[2], dtype=int)
        else:
            bond_features = [BOND_PROP2OH[k].get(v, 0) for k, v in
                             zip(ORDERED_BOND_OH_KEYS, [self.bond_type, self.bond_stereo, 1])]
            adj[self.atom_map1, self.atom_map2] = bond_features
            adj[self.atom_map2, self.atom_map1] = bond_features
        return adj, nodes
    
    def __str__(self):
        if self.bond_type is None:
            return f"DeleteBond({self.atom_map1}-{self.atom_map2})"
        bt_names = {1: 'SINGLE', 2: 'DOUBLE', 3: 'TRIPLE', 12: 'AROMATIC'}
        bt = bt_names.get(self.bond_type, str(self.bond_type))
        return f"BondEdit({self.atom_map1}-{self.atom_map2}): {bt}"


class AddAtomAction(ReactionAction):
    """添加新原子及连接键"""
    def __init__(self, atom_map1, atom_map2, bond_type, bond_stereo,
                 atomic_num, formal_charge, chiral_tag, num_explicit_hs, is_aromatic, is_hard=False):
        super().__init__(atom_map1, atom_map2, is_hard)
        self.bond_type = bond_type
        self.bond_stereo = bond_stereo
        self.atomic_num = atomic_num
        self.formal_charge = formal_charge
        self.chiral_tag = chiral_tag
        self.num_explicit_hs = num_explicit_hs
        self.is_aromatic = is_aromatic
    
    def get_tuple(self):
        return ('add_atom', ((self.bond_type, self.bond_stereo),
                (self.atomic_num, self.formal_charge, self.chiral_tag, self.num_explicit_hs, self.is_aromatic)))
    
    def apply(self, mol):
        atom_ind = get_atom_ind(mol, self.atom_map1)
        new_a = Chem.Atom(self.atomic_num)
        new_a.SetFormalCharge(self.formal_charge)
        new_a.SetChiralTag(rdchem.ChiralType.values[self.chiral_tag])
        new_a.SetNumExplicitHs(self.num_explicit_hs)
        new_a.SetIsAromatic(self.is_aromatic)
        new_a.SetAtomMapNum(self.atom_map2)
        new_a.SetBoolProp('is_edited', True)
        
        old_atom = mol.GetAtomWithIdx(atom_ind)
        if old_atom.HasProp('in_reactant'):
            new_a.SetBoolProp('in_reactant', old_atom.GetBoolProp('in_reactant'))
        
        num_atoms = mol.GetNumAtoms()
        mol.AddAtom(new_a)
        b_type = rdchem.BondType.values[self.bond_type]
        bond_idx = mol.AddBond(atom_ind, num_atoms, order=b_type) - 1
        bond = mol.GetBondWithIdx(bond_idx)
        bond.SetStereo(rdchem.BondStereo.values[self.bond_stereo])
        bond.SetBoolProp('is_edited', True)
        return mol
    
    def graph_apply(self, adj, nodes):
        max_num = max(len(adj), self.atom_map2 + 1)
        new_adj = np.zeros((max_num, max_num, adj.shape[2]), dtype=int)
        new_adj[:adj.shape[0], :adj.shape[1]] = adj
        new_nodes = np.zeros((max_num, nodes.shape[1]), dtype=int)
        new_nodes[:nodes.shape[0]] = nodes
        
        # 新原子特征
        new_a = Chem.Atom(self.atomic_num)
        new_a.SetFormalCharge(self.formal_charge)
        new_a.SetChiralTag(rdchem.ChiralType.values[self.chiral_tag])
        new_a.SetNumExplicitHs(self.num_explicit_hs)
        new_a.SetIsAromatic(self.is_aromatic)
        new_nodes[self.atom_map2] = get_atom_features(new_a, ORDERED_ATOM_OH_KEYS, atom_prop2oh=ATOM_PROP2OH)
        
        # 新原子-旧原子键
        bond_features = [BOND_PROP2OH[k].get(v, 0) for k, v in
                         zip(ORDERED_BOND_OH_KEYS, [self.bond_type, self.bond_stereo, 1])]
        new_adj[self.atom_map1, self.atom_map2] = bond_features
        new_adj[self.atom_map2, self.atom_map1] = bond_features
        
        # 超节点-新原子边 + 自环
        new_adj[0, self.atom_map2, 0] = BOND_PROP2OH['bond_type']['supernode']
        new_adj[self.atom_map2, 0, 0] = BOND_PROP2OH['bond_type']['supernode']
        new_adj[self.atom_map2, self.atom_map2, 0] = BOND_PROP2OH['bond_type']['self']
        
        return new_adj, new_nodes
    
    def __str__(self):
        from rdkit.Chem.rdchem import GetPeriodicTable
        symbol = GetPeriodicTable().GetElementSymbol(self.atomic_num)
        return f"AddAtom({symbol}:{self.atom_map2} to {self.atom_map1})"


print("✅ 5 种图编辑动作类定义完成:")
print("  - StopAction: 终止编辑")
print("  - AtomEditAction: 修改原子属性")
print("  - BondEditAction: 修改/添加/删除化学键")
print("  - AddAtomAction: 添加新原子")
print("  - (AddRingAction: 完整实现中包含苯环添加，此处省略)")
print(f"\n动作词汇表中的原子特征索引: {action_vocab['atom_feat_ind']}")
print(f"动作词汇表中的键特征索引: {action_vocab['bond_feat_ind']}")

动作词汇表键:
  prop2oh: ['atom', 'bond']
  atom_feat_ind: ['is_supernode', 'atomic_num', 'formal_charge', 'chiral_tag', 'num_explicit_hs', 'is_aromatic', 'is_edited', 'is_reactant']
  bond_feat_ind: ['bond_type', 'bond_stereo', 'is_edited']
✅ 5 种图编辑动作类定义完成:
  - StopAction: 终止编辑
  - AtomEditAction: 修改原子属性
  - BondEditAction: 修改/添加/删除化学键
  - AddAtomAction: 添加新原子
  - (AddRingAction: 完整实现中包含苯环添加，此处省略)

动作词汇表中的原子特征索引: {'is_supernode': 0, 'atomic_num': 1, 'formal_charge': 2, 'chiral_tag': 3, 'num_explicit_hs': 4, 'is_aromatic': 5, 'is_edited': 6, 'is_reactant': 7}
动作词汇表中的键特征索引: {'bond_type': 0, 'bond_stereo': 1, 'is_edited': 2}


### Step 5.2: 对比产物与反应物，提取编辑序列

核心算法 `generate_action()` 的逻辑：

1. 遍历产物和反应物的**所有原子**，找出属性差异 → `changed_atoms`
2. 遍历产物和反应物的**所有化学键**，找出：
   - 被删除的键 (`deleted_bonds`)
   - 被添加的键 (`added_bonds`)  
   - 被修改的键 (`changed_bonds`)
3. 找出仅存在于反应物中的原子 → `new_atoms`
4. 按优先级选出**下一个动作**
5. **执行该动作**（更新分子和图），回到步骤 1
6. 直到没有差异 → 发出 `StopAction`

In [9]:
# ============================================================
# Step 5.2: 动作提取与执行（简化版 ReactionSampleGenerator）
# ============================================================

def mark_reactants(source_mol, target_mol):
    """标记 source_mol 中哪些原子也出现在 target_mol（即产物中的原子）"""
    target_atoms = set(a.GetAtomMapNum() for a in target_mol.GetAtoms())
    for a in source_mol.GetAtoms():
        m = a.GetAtomMapNum()
        if m is not None and m > 0 and m in target_atoms:
            a.SetBoolProp('in_target', True)


def generate_action(source_mol, target_mol, edited_atoms):
    """
    比较 source_mol（当前状态）和 target_mol（目标状态），返回下一个动作。
    
    这里 source_mol 从产物开始，target_mol 是反应物。
    逆合成：产物 → 反应物
    """
    source_atoms = {}
    target_atoms = {}
    source_bonds = {}
    target_bonds = {}
    atoms_only_in_target = {}
    target_atomic_nums = {}
    
    # 收集 source 原子
    for a in source_mol.GetAtoms():
        source_atoms[a.GetAtomMapNum()] = atom_to_edit_tuple(a)
    
    # 收集 target 原子，找出差异
    changed_atoms = {}
    for a in target_mol.GetAtoms():
        am = a.GetAtomMapNum()
        at = atom_to_edit_tuple(a)
        target_atoms[am] = at
        target_atomic_nums[am] = a.GetAtomicNum()
        if am not in source_atoms:
            atoms_only_in_target[am] = a
        elif source_atoms[am] != at:
            changed_atoms[am] = AtomEditAction(am, *at, is_hard=am not in edited_atoms)
    
    # 收集 source 键
    for bond in source_mol.GetBonds():
        bt = get_bond_tuple(bond)
        source_bonds[(bt[0], bt[1])] = bt[2:]
    
    # 收集 target 键
    deleted_bonds = {}
    added_bonds = {}
    changed_bonds = {}
    new_atoms = {}
    
    for bond in target_mol.GetBonds():
        bt = get_bond_tuple(bond)
        a1, a2, bond_type = bt[0], bt[1], bt[2:]
        target_bonds[(a1, a2)] = bond_type
        
        # 检查是否有新原子需要添加
        for src_a, new_a in [(a1, a2), (a2, a1)]:
            if new_a in atoms_only_in_target and src_a not in atoms_only_in_target:
                new_atoms[(new_a, src_a)] = AddAtomAction(
                    src_a, new_a, *bond_type,
                    target_atomic_nums[new_a], *target_atoms[new_a],
                    is_hard=src_a not in edited_atoms)
        
        # 检查是否有新键
        if a1 in source_atoms and a2 in source_atoms and (a1, a2) not in source_bonds:
            added_bonds[(a2, a1)] = BondEditAction(a1, a2, *bond_type, 
                                                    is_hard=a1 not in edited_atoms and a2 not in edited_atoms)
    
    # 检查删除和修改的键
    for bond_atoms, bond_type in source_bonds.items():
        if bond_atoms not in target_bonds:
            if bond_atoms[0] in target_atoms or bond_atoms[1] in target_atoms:
                deleted_bonds[(bond_atoms[1], bond_atoms[0])] = BondEditAction(
                    bond_atoms[0], bond_atoms[1], None, None,
                    is_hard=bond_atoms[0] not in edited_atoms and bond_atoms[1] not in edited_atoms)
        elif target_bonds[bond_atoms] != bond_type:
            changed_bonds[(bond_atoms[1], bond_atoms[0])] = BondEditAction(
                bond_atoms[0], bond_atoms[1], *target_bonds[bond_atoms],
                is_hard=bond_atoms[0] not in edited_atoms and bond_atoms[1] not in edited_atoms)
    
    # 按逆合成优先级排序（先删键，再加键，再改键，再改原子，最后加原子）
    action_type_priorities = [
        ('double', deleted_bonds),
        ('double', added_bonds),
        ('double', changed_bonds),
        ('single', changed_atoms),
        ('double', new_atoms),
    ]
    
    target_atom_keys = sorted(target_atoms.keys())
    
    for atom_map1 in target_atom_keys:
        for action_type, actions_dict in action_type_priorities:
            if action_type == 'double':
                for atom_map2 in target_atom_keys:
                    if (atom_map1, atom_map2) in actions_dict:
                        return actions_dict[(atom_map1, atom_map2)]
                    elif (atom_map2, atom_map1) in actions_dict:
                        return actions_dict[(atom_map2, atom_map1)]
            elif atom_map1 in actions_dict:
                return actions_dict[atom_map1]
    
    # 按优先级兜底
    for action_type, actions_dict in action_type_priorities:
        if len(actions_dict) > 0:
            key = sorted(actions_dict.keys())[0]
            return actions_dict[key]
    
    return StopAction()


def gen_training_samples(product_smi, reactant_smi, n_max_steps=32):
    """
    完整的训练样本生成流程：
    1. 解析 SMILES
    2. 预处理（修复映射、规范化）
    3. 迭代生成动作序列
    """
    target_mol = Chem.MolFromSmiles(reactant_smi)
    source_mol = Chem.MolFromSmiles(product_smi)
    
    if target_mol is None or source_mol is None:
        return [], ''
    
    # 预处理
    target_mol, source_mol = fix_incomplete_mappings(target_mol, source_mol)
    target_mol, source_mol = reac_to_canonical(target_mol, source_mol)
    source_mol = fix_explicit_hs(source_mol)
    target_mol = fix_explicit_hs(target_mol)
    source_mol = renumber_atoms_for_mapping(source_mol)
    target_mol = renumber_atoms_for_mapping(target_mol)
    
    # 标记反应物
    source_mol = Chem.RWMol(source_mol)
    mark_reactants(source_mol, target_mol)
    
    training_samples = []
    edited_atoms = set()
    
    for step in range(n_max_steps):
        source_mol.UpdatePropertyCache(strict=False)
        
        # 生成动作
        action = generate_action(source_mol, target_mol, edited_atoms)
        
        # 获取当前图表示
        adj, nodes = get_graph(source_mol, ravel=False, to_array=True)
        
        sample = {
            'step': step,
            'action': action,
            'action_tuple': action.get_tuple(),
            'action_str': str(action),
            'adj': adj,
            'nodes': nodes,
            'atom_map1': action.atom_map1,
            'atom_map2': action.atom_map2 if isinstance(action, BondEditAction) else -1,
            'is_hard': action.is_hard,
        }
        training_samples.append(sample)
        
        # 更新 edited_atoms
        if action.atom_map1 > 0:
            edited_atoms.add(action.atom_map1)
        if action.atom_map2 > 0:
            edited_atoms.add(action.atom_map2)
        
        if isinstance(action, StopAction):
            final_smi = Chem.MolToSmiles(source_mol)
            return training_samples, final_smi
        
        # 执行动作，更新分子
        source_mol = action.apply(source_mol)
    
    return [], ''  # 超过最大步数


# === 演示：对一个反应提取编辑序列 ===
print("=" * 60)
print("Step 5: 图编辑动作提取演示")
print("=" * 60)

# 取 demo 数据中的第一个反应
demo_df = pd.read_csv('data/demo_data.csv')
rxn = demo_df.iloc[0]['reactants>reagents>production']
parts = rxn.split('>>')
product_smi = parts[0]
reactant_smi = parts[1]

print(f"\n反应: {rxn[:80]}...")
print(f"产物:  {product_smi[:60]}...")
print(f"反应物: {reactant_smi[:60]}...")

# 生成训练样本
samples, final_smi = gen_training_samples(product_smi, reactant_smi)

print(f"\n共提取 {len(samples)} 个编辑步骤:")
print("-" * 50)
for s in samples:
    tag = "⭐" if s['is_hard'] else "  "
    nodes_shape = s['nodes'].shape
    print(f"  {tag} Step {s['step']}: {s['action_str']}")
    print(f"       图状态: {nodes_shape[0]} 节点, {np.count_nonzero(s['adj'][:,:,0])} 条边条目")

print(f"\n最终 SMILES: {final_smi}")
print(f"动作类型分布:")
from collections import Counter
action_types = [s['action_tuple'][0] for s in samples]
for at, cnt in Counter(action_types).items():
    print(f"  {at}: {cnt}")

Step 5: 图编辑动作提取演示

反应: C1CCOC1.CCOCC.Cc1ccccc1.O=[CH:10][CH:9]1[C:7](=[O:8])[c:5]2[c:4]([n:3][c:2]([CH3...
产物:  C1CCOC1.CCOCC.Cc1ccccc1.O=[CH:10][CH:9]1[C:7](=[O:8])[c:5]2[...
反应物: [CH3:1][c:2]1[n:3][c:4]2[c:5]([s:6]1)[C:7](=[O:8])/[C:9](=[C...

共提取 4 个编辑步骤:
--------------------------------------------------
  ⭐ Step 0: BondEdit(14-15): DOUBLE
       图状态: 34 节点, 164 条边条目
     Step 1: BondEdit(15-31): SINGLE
       图状态: 34 节点, 164 条边条目
     Step 2: DeleteBond(15-16)
       图状态: 34 节点, 166 条边条目
     Step 3: Stop
       图状态: 34 节点, 164 条边条目

最终 SMILES: [CH2:1]1[CH2:2][CH2:3][O:4][CH2:5]1.[CH3:19][CH2:20][O:21][CH2:22][CH3:23].[CH3:24][c:25]1[cH:26][cH:27][cH:28][cH:29][cH:30]1.[CH3:6][c:7]1[n:8][c:9]2[c:10]([s:11]1)[C:12](=[O:13])[C:14](=[CH:15][N:31]([CH3:32])[CH3:33])[CH2:17][CH2:18]2.[OH2:16]
动作类型分布:
  change_bond: 2
  delete_bond: 1
  stop: 1


## Step 6: 批量处理与稀疏矩阵存储

实际训练中，MEGAN 需要处理数万条反应。数据处理的最终输出是**稀疏矩阵 (Sparse Matrix)**，原因是：

- 每个训练样本的分子图大部分位置为 0（稀疏性高）
- 不同样本的图大小不同，需要统一到 `max_n_nodes` 维度
- 使用 `scipy.sparse.csr_matrix` 可以极大节省内存

### 存储格式
```
nodes_mat:  (n_samples, max_n_nodes)      -- 每行是一个样本的节点特征（raveled）
adj_mat:    (n_samples, max_n_nodes²)     -- 每行是邻接矩阵展平后的稀疏向量
metadata:   DataFrame                      -- 反应索引、步数、最终SMILES等
actions:    List[Tuple]                    -- 每步的动作元组
```

In [11]:
# ============================================================
# Step 6: 批量处理 demo 数据，转为稀疏矩阵存储
# ============================================================
from scipy import sparse

N_MAX_STEPS = 32  # 每个反应最多编辑步数
MAX_N_NODES = 64  # 最大节点数（统一维度）

print("=" * 60)
print("Step 6: 批量处理 demo 数据")
print("=" * 60)

# 存储容器
all_nodes_rows, all_nodes_cols, all_nodes_vals = [], [], []
all_adj_rows, all_adj_cols, all_adj_vals = [], [], []
all_actions = []

metadata = {
    'reaction_ind': [],
    'n_samples': [],
    'start_ind': [],
    'final_smi': [],
    'product_smi': [],
}

n_success = 0
n_fail = 0
total_samples = 0

for rxn_idx, row in demo_df.iterrows():
    rxn = row['reactants>reagents>production']
    parts = rxn.split('>>')
    product_smi = parts[0]
    reactant_smi = parts[1]
    
    samples, final_smi = gen_training_samples(product_smi, reactant_smi, n_max_steps=N_MAX_STEPS)
    
    if len(samples) == 0:
        n_fail += 1
        continue
    
    n_success += 1
    start_ind = rxn_idx * N_MAX_STEPS
    
    metadata['reaction_ind'].append(rxn_idx)
    metadata['n_samples'].append(len(samples))
    metadata['start_ind'].append(start_ind)
    metadata['final_smi'].append(final_smi)
    metadata['product_smi'].append(product_smi)
    
    for sample_ind, sample in enumerate(samples):
        ind = start_ind + sample_ind
        all_actions.append((ind, sample['action_tuple']))
        
        # 节点特征 → raveled 后存入稀疏矩阵
        adj_arr, nodes_arr = sample['adj'], sample['nodes']
        
        # ravel 节点特征
        node_oh_dim = [len(ATOM_PROP2OH[k]) + 1 for k in ORDERED_ATOM_OH_KEYS if k in ATOM_PROP2OH]
        for j in range(min(len(nodes_arr), MAX_N_NODES)):
            raveled = ravel_atom_features(nodes_arr[j], node_oh_dim)
            if raveled != 0:
                all_nodes_rows.append(ind)
                all_nodes_cols.append(j)
                all_nodes_vals.append(raveled)
        
        # 邻接矩阵 → 展平后存入稀疏矩阵
        edge_oh_dim = [len(BOND_PROP2OH[k]) + 1 for k in ORDERED_BOND_OH_KEYS if k in BOND_PROP2OH]
        n = min(adj_arr.shape[0], MAX_N_NODES)
        for r in range(n):
            for c in range(n):
                if np.any(adj_arr[r, c] != 0):
                    raveled = ravel_bond_features(adj_arr[r, c], edge_oh_dim)
                    if raveled != 0:
                        all_adj_rows.append(ind)
                        all_adj_cols.append(r * MAX_N_NODES + c)
                        all_adj_vals.append(raveled)
    
    total_samples += len(samples)
    
    # 进度显示
    if (rxn_idx + 1) % 10 == 0:
        print(f"  已处理 {rxn_idx + 1} / {len(demo_df)} 条反应...")

print(f"\n处理完成!")
print(f"  成功: {n_success}, 失败: {n_fail}")
print(f"  总训练样本数: {total_samples}")

# 构建稀疏矩阵
n_total_slots = len(demo_df) * N_MAX_STEPS

nodes_mat = sparse.csr_matrix(
    (all_nodes_vals, (all_nodes_rows, all_nodes_cols)),
    shape=(n_total_slots, MAX_N_NODES)
)

adj_mat = sparse.csr_matrix(
    (all_adj_vals, (all_adj_rows, all_adj_cols)),
    shape=(n_total_slots, MAX_N_NODES ** 2)
)

metadata_df = pd.DataFrame(metadata)

print(f"\n稀疏矩阵统计:")
print(f"  nodes_mat: shape={nodes_mat.shape}, nnz={nodes_mat.nnz}, "
      f"稀疏率={1 - nodes_mat.nnz / (nodes_mat.shape[0] * nodes_mat.shape[1]):.4f}")
print(f"  adj_mat:   shape={adj_mat.shape}, nnz={adj_mat.nnz}, "
      f"稀疏率={1 - adj_mat.nnz / (adj_mat.shape[0] * adj_mat.shape[1]):.6f}")

print(f"\nmetadata 前5行:")
print(metadata_df.head())

print(f"\n动作类型总分布:")
all_action_types = [a[1][0] for a in all_actions]
for at, cnt in Counter(all_action_types).items():
    print(f"  {at}: {cnt} ({cnt/len(all_action_types)*100:.1f}%)")

Step 6: 批量处理 demo 数据
  已处理 10 / 1000 条反应...
  已处理 20 / 1000 条反应...
  已处理 30 / 1000 条反应...
  已处理 40 / 1000 条反应...
  已处理 50 / 1000 条反应...
  已处理 60 / 1000 条反应...
  已处理 70 / 1000 条反应...
  已处理 80 / 1000 条反应...
  已处理 90 / 1000 条反应...
  已处理 100 / 1000 条反应...
  已处理 110 / 1000 条反应...
  已处理 120 / 1000 条反应...
  已处理 130 / 1000 条反应...
  已处理 140 / 1000 条反应...
  已处理 150 / 1000 条反应...
  已处理 160 / 1000 条反应...
  已处理 170 / 1000 条反应...
  已处理 180 / 1000 条反应...
  已处理 190 / 1000 条反应...
  已处理 200 / 1000 条反应...
  已处理 210 / 1000 条反应...
  已处理 220 / 1000 条反应...
  已处理 230 / 1000 条反应...
  已处理 240 / 1000 条反应...
  已处理 250 / 1000 条反应...
  已处理 260 / 1000 条反应...
  已处理 270 / 1000 条反应...
  已处理 280 / 1000 条反应...
  已处理 290 / 1000 条反应...
  已处理 300 / 1000 条反应...
  已处理 310 / 1000 条反应...
  已处理 320 / 1000 条反应...
  已处理 330 / 1000 条反应...
  已处理 340 / 1000 条反应...
  已处理 350 / 1000 条反应...
  已处理 360 / 1000 条反应...
  已处理 370 / 1000 条反应...
  已处理 380 / 1000 条反应...
  已处理 390 / 1000 条反应...
  已处理 400 / 1000 条反应...
  已处理 410 / 1000 条反应...
  已处

## 总结

本 Notebook 完成了 MEGAN 数据处理的完整流程：

| 步骤 | 内容 | 关键函数/概念 |
|------|------|--------------|
| Step 1 | 加载 CSV 数据 | `pd.read_csv`, 反应 SMILES 解析 |
| Step 2 | SMILES 预处理 | `fix_incomplete_mappings`, `reac_to_canonical`, `fix_explicit_hs` |
| Step 3 | 原子/键特征编码 | `get_atom_features`, `get_bond_features`, One-Hot 字典 |
| Step 4 | 分子图构建 | `get_graph`, Supernode, 自环, COO/密集格式 |
| Step 5 | 图编辑动作提取 | `generate_action`, 5种动作类, 优先级排序 |
| Step 6 | 批量处理存储 | `scipy.sparse`, 稀疏矩阵, metadata |

### 核心理念
MEGAN 的数据处理将每个化学反应转化为一系列**图编辑操作**（动作序列），这使得模型可以通过**自回归方式**逐步生成反应物——这是 MEGAN 与传统模板方法和 SMILES 序列方法的本质区别。

### 下一步
在 `3_模型展示.ipynb` 中，我们将学习 MEGAN 如何利用这些训练样本，通过 GAT 编码器和动作解码器进行逆合成预测。